#### Ingest drivers.json file
1. Read the file using dataframe reader API
2. Add metadata columns: source file, ingestion timestamp
3. Write to bronze delta table

In [0]:
%run ../Workspace/common/configuration_environment

In [0]:
%run ../Workspace/common/bronze_helpers

In [0]:
# Define source file and table name
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

##### 1. Read JSON file using dataframe reader API

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DataType

name_schema = StructType([
    StructField("givenName", StringType()),
    StructField("familyName", StringType())
])

drivers_schema = StructType(fields=[ 
    StructField("_corrupt_record", StringType()),
    StructField("constructorId", StringType()),
#    StructField("dateOfBirth", DataType()), # It throws errors here
    StructField("dateOfBirth", StringType()),
    StructField("driverId", StringType()),
    StructField("name", name_schema),
    StructField("nationality", StringType()),
    StructField("url", StringType()),
])

In [0]:
drivers_df = (
    spark.read
        .format('json')
        .schema(drivers_schema)
        .option('mode', 'FAILFAST')
#        .option('inferSchema', True)
        .load(source_file)
)

In [0]:
display(drivers_df)

_corrupt_record,constructorId,dateOfBirth,driverId,name,nationality,url
null,null,1932-07-10,abate,"List(carlo, abate)",italian,http://en.wikipedia.org/wiki/Carlo_Mario_Abate
null,null,1913-03-21,abecassis,"List(george, abecassis)",british,http://en.wikipedia.org/wiki/George_Abecassis
null,null,1957-11-27,acheson,"List(kenny, acheson)",british,http://en.wikipedia.org/wiki/Kenny_Acheson
null,null,1969-11-19,adams,"List(philippe, adams)",belgian,http://en.wikipedia.org/wiki/Philippe_Adams
null,null,1913-12-15,ader,"List(walt, ader)",american,http://en.wikipedia.org/wiki/Walt_Ader
null,null,1921-11-05,adolff,"List(kurt, adolff)",german,http://en.wikipedia.org/wiki/Kurt_Adolff
null,null,1913-08-21,agabashian,"List(fred, agabashian)",american,http://en.wikipedia.org/wiki/Fred_Agabashian
null,null,1940-04-19,ahrens,"List(kurt, ahrens)",german,"http://en.wikipedia.org/wiki/Kurt_Ahrens,_Jr."
null,null,1995-09-23,aitken,"List(jack, aitken)",british,http://en.wikipedia.org/wiki/Jack_Aitken
null,null,1979-04-16,albers,"List(christijan, albers)",dutch,http://en.wikipedia.org/wiki/Christijan_Albers


##### 2. Add metadata columns: source file, ingestion timestamp

In [0]:
drivers_final = add_ingestion_metadata(drivers_df)

In [0]:
display(drivers_final)

_corrupt_record,constructorId,dateOfBirth,driverId,name,nationality,url,ingestion_timestamp,source_file
null,null,1932-07-10,abate,"List(carlo, abate)",italian,http://en.wikipedia.org/wiki/Carlo_Mario_Abate,2026-04-27T13:14:10.165259Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1913-03-21,abecassis,"List(george, abecassis)",british,http://en.wikipedia.org/wiki/George_Abecassis,2026-04-27T13:14:10.165259Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1957-11-27,acheson,"List(kenny, acheson)",british,http://en.wikipedia.org/wiki/Kenny_Acheson,2026-04-27T13:14:10.165259Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1969-11-19,adams,"List(philippe, adams)",belgian,http://en.wikipedia.org/wiki/Philippe_Adams,2026-04-27T13:14:10.165259Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1913-12-15,ader,"List(walt, ader)",american,http://en.wikipedia.org/wiki/Walt_Ader,2026-04-27T13:14:10.165259Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1921-11-05,adolff,"List(kurt, adolff)",german,http://en.wikipedia.org/wiki/Kurt_Adolff,2026-04-27T13:14:10.165259Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1913-08-21,agabashian,"List(fred, agabashian)",american,http://en.wikipedia.org/wiki/Fred_Agabashian,2026-04-27T13:14:10.165259Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1940-04-19,ahrens,"List(kurt, ahrens)",german,"http://en.wikipedia.org/wiki/Kurt_Ahrens,_Jr.",2026-04-27T13:14:10.165259Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1995-09-23,aitken,"List(jack, aitken)",british,http://en.wikipedia.org/wiki/Jack_Aitken,2026-04-27T13:14:10.165259Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1979-04-16,albers,"List(christijan, albers)",dutch,http://en.wikipedia.org/wiki/Christijan_Albers,2026-04-27T13:14:10.165259Z,dbfs:/Volumes/formula1/landing/files/drivers.json


##### 3. Write to bronze delta table

In [0]:
(
    drivers_final
        .write
        .mode('overwrite')
        .format('delta')
#        .saveAsTable('table_name') # doesn't work with overwrite mode
        .saveAsTable('formula1.bronze.drivers')
)

In [0]:
display(spark.table(table_name))

_corrupt_record,constructorId,dateOfBirth,driverId,name,nationality,url,ingestion_timestamp,source_file
null,null,1932-07-10,abate,"List(carlo, abate)",italian,http://en.wikipedia.org/wiki/Carlo_Mario_Abate,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1913-03-21,abecassis,"List(george, abecassis)",british,http://en.wikipedia.org/wiki/George_Abecassis,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1957-11-27,acheson,"List(kenny, acheson)",british,http://en.wikipedia.org/wiki/Kenny_Acheson,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1969-11-19,adams,"List(philippe, adams)",belgian,http://en.wikipedia.org/wiki/Philippe_Adams,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1913-12-15,ader,"List(walt, ader)",american,http://en.wikipedia.org/wiki/Walt_Ader,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1921-11-05,adolff,"List(kurt, adolff)",german,http://en.wikipedia.org/wiki/Kurt_Adolff,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1913-08-21,agabashian,"List(fred, agabashian)",american,http://en.wikipedia.org/wiki/Fred_Agabashian,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1940-04-19,ahrens,"List(kurt, ahrens)",german,"http://en.wikipedia.org/wiki/Kurt_Ahrens,_Jr.",2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1995-09-23,aitken,"List(jack, aitken)",british,http://en.wikipedia.org/wiki/Jack_Aitken,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json
null,null,1979-04-16,albers,"List(christijan, albers)",dutch,http://en.wikipedia.org/wiki/Christijan_Albers,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json


In [0]:
%sql
SELECT season, COUNT(*) FROM formula1.bronze.results GROUP BY season ORDER BY season;

season,count(1)
null,5
1950,160
1951,179
1952,204
1953,246
1954,229
1955,177
1956,189
1957,166
1958,226
